# 1 — Build the Graph

Three layers, built in order. Each has different provenance, different cost, and a
different failure mode.

```
  1  structure   Document → Section → Chunk        free, exact, from metadata
  2  registry    Trial and everything it links     free, authoritative, from the API
  3  extracted   protocol operational detail       one LLM call per chunk
```

The order matters. Structure first because it is free and gives everything else a
place to attach. Registry second because it is ground truth. Extraction last, and
only for what the other two cannot know.

Every node carries a `source` property. A query can then demand facts or accept
claims — and because the registry is ground truth for the fields it covers,
extraction accuracy becomes measurable rather than assumed.

| § | Stage |
|---|---|
| 0 | Setup |
| 1 | Layer 1 — document structure |
| 2 | Layer 2 — registry facts |
| 3 | The join |
| 4 | Layer 3 — what only the document knows |
| 5 | Extract and load |
| 6 | Score the extractor against ground truth |

---
## 0. Setup

In [ ]:
# MUST run before any `from graph_rag import ...` below. config.py reads
# every setting with os.getenv(...) at MODULE level, the moment it is
# imported — not lazily, inside a function. If .env loads after that
# import has already happened, every setting is frozen at whatever was in
# the environment before .env was read, usually empty or a placeholder.
from dotenv import load_dotenv

load_dotenv()


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

# Everything else comes from .env via load_dotenv() above. This checks only
# what actually has no usable fallback:
#
#   OPENAI_API_KEY, PINECONE_API_KEY   no default exists — os.environ[...]
#                                       raises a bare KeyError deep inside
#                                       the first API call that needs it
#   NEO4J_PASSWORD                     defaults to "", which store.driver()
#                                       already rejects with a clear error —
#                                       checked here too, so the notebook
#                                       fails on THIS cell, not several cells
#                                       and a stack trace later
#
# NEO4J_URI, NEO4J_USER, NEO4J_DATABASE, INDEX_NAME and the rest all have
# working defaults already, in config.py itself (local Docker, "neo4j",
# "rag-docs") — restating the same defaults here would just be a second
# place for them to drift out of agreement with the first. Override any of
# them in .env, not here.
missing = [name for name in ("OPENAI_API_KEY", "PINECONE_API_KEY", "NEO4J_PASSWORD")
          if not os.getenv(name)]
if missing:
    raise RuntimeError(f"missing from .env (or the environment): {', '.join(missing)}")
print("required secrets present")


In [ ]:
from graph_rag import accuracy, config, extract, registry, schema, store, structure
from graph_rag import chunks as chunk_reader
import pandas as pd

# A live, verified connection — store.driver() checks the URI scheme and
# connectivity itself, so a bad password or a plain bolt:// URI against Aura
# fails right here with a clear reason, not three cells later inside a
# transaction error that reads like a Cypher problem.
driver = store.driver()
session = driver.session(database=config.NEO4J_DATABASE)

index = chunk_reader.index()

# Scope the build to one document while iterating on the schema — its doc_id,
# the prefix chunk ids were written with. None reads every document already in
# the vector index, which is what a multi-document graph wants.
DOC_ID = None

print(f"connected to {config.NEO4J_URI}")
print(f"reading from index {config.INDEX_NAME!r}")


In [ ]:
# Start clean while iterating on the schema. Comment out to add documents to an
# existing graph — every write below is MERGE, so re-running converges rather than
# duplicating.
store.clear(session)

structure.create_constraints(session)
registry.create_constraints(session)
store.create_constraints(session)
print("constraints in place")

---
## 1. Layer 1 — document structure

Free. No LLM call, nothing to hallucinate. It comes entirely from metadata the
ingestion pipeline already wrote.

```
(:Document)-[:HAS_SECTION]->(:Section)-[:HAS_CHUNK]->(:Chunk)
(:Chunk)-[:NEXT]->(:Chunk)
```

Built first because it is exact by construction and gives every extracted claim in
layer 3 a path back to the page it came from.

In [ ]:
records = chunk_reader.load_chunks(index, DOC_ID)
print(f"{len(records)} chunks across "
      f"{len({c.get('doc_id') for c in records})} documents\n")

counts = structure.load_structure(session, records)

`linked_to_trial` is the number that matters. It counts documents whose NCT number
was found **and** whose trial is already in the registry — but the registry has not
been loaded yet, so this will be 0 on a first run. It is populated in §3.

In [ ]:
# How the NCT number is found. No model involved: an NCT id has fixed form, so a
# regex reads it faster, free, and without the ability to invent one.
for doc_id, source in [("nct04368728-remdesivir-covid", ""),
                       ("remdesivir-covid", "NCT04368728_Remdesivir_COVID.pdf"),
                       ("ai-enablers-report", "AI-Enablers.pdf")]:
    print(f"{doc_id:<34}{source:<36}-> {structure.find_nct_id(doc_id, source) or '(none)'}")

Note the second row. The pattern uses lookarounds rather than `\b`, because an
underscore is a word character — so `\bNCT\d{8}\b` does **not** match inside
`NCT04368728_Remdesivir_COVID.pdf`, which is exactly how these files are named. With
`\b` the join fails on every document, and fails silently: each one simply ends up
unlinked.

---
## 2. Layer 2 — registry facts

Also free, and authoritative. ClinicalTrials.gov records sponsor, phase, conditions,
interventions, sites and outcomes, so extracting them from a PDF would mean paying a
model to guess at facts that are correct by definition.

Everything written here carries `source: "registry"`.

In [ ]:
# Gated by RUN_REGISTRY (config.py) — on by default, since this costs nothing
# but a public API call. Set RUN_REGISTRY=0 to skip it entirely, e.g. for a
# corpus that names no real, registered trials.
if config.RUN_REGISTRY:
    # NCT numbers taken from the documents themselves rather than a hardcoded
    # list, so the registry layer covers exactly the corpus that was ingested.
    nct_ids = sorted({structure.find_nct_id(c.get("doc_id", ""), c.get("source", ""),
                                            c.get("text", ""))
                      for c in records} - {None})
    print(f"{len(nct_ids)} trials referenced by the corpus\n")

    result = registry.load_trials(session, nct_ids)
else:
    print("RUN_REGISTRY=0 — skipping the registry layer. Document/Section/Chunk "
          "nodes exist regardless; no Trial, Sponsor, Drug or other registry "
          "node will, and structure.load_structure()'s ABOUT edges below will "
          "correctly link nothing rather than nothing existing to link to.")


In [ ]:
# MeSH terms are the reason this layer needs no fuzzy entity matching. They are
# NLM's controlled vocabulary, so two trials describing the same condition
# differently connect through a shared node without anyone guessing.
pd.DataFrame([dict(r) for r in session.run("""
    MATCH (m:MeSHTerm)<-[:INDEXED_AS]-(t:Trial)
    RETURN m.term AS mesh_term, count(t) AS trials,
           collect(t.nctId)[..4] AS sample
    ORDER BY trials DESC LIMIT 15
""")])

---
## 3. The join

`(:Document)-[:ABOUT]->(:Trial)`. Without it these are two separate graphs sharing a
database. With it, a traversal can start from a registry fact and end at the passage
of the protocol that discusses it.

Re-running the structure loader now links them, because the trials exist.

In [ ]:
counts = structure.load_structure(session, records, verbose=False)
print(f"documents linked to a trial: {counts['linked_to_trial']}/{counts['documents']}")
if not config.RUN_REGISTRY:
    print("(RUN_REGISTRY=0 — 0 is expected here, not a failure: there are no "
          "Trial nodes for this MATCH to find yet.)")

pd.DataFrame([dict(r) for r in session.run("""
    MATCH (d:Document)-[:ABOUT]->(t:Trial)
    RETURN d.docId AS document, t.nctId AS trial, t.phase AS phase,
           t.overallStatus AS status, d.nChunks AS chunks
    ORDER BY document LIMIT 25
""")])


---
## 4. Layer 3 — what only the document knows

The extraction schema is deliberately small. Everything the registry owns is
excluded, and the prompt says so explicitly — so the model is not competing with
ground truth.

In [ ]:
print(f"REGISTRY OWNS ({len(schema.REGISTRY_OWNED)}) — never extracted")
print("  " + ", ".join(sorted(schema.REGISTRY_OWNED)))

print(f"\nEXTRACTED ({len(schema.ENTITY_TYPES)}) — only in the document")
for name, description in schema.ENTITY_TYPES.items():
    print(f"  {name:<12} {description}")

print(f"\nRELATIONSHIPS ({len(schema.RELATION_TYPES)})")
for name, (source, target) in schema.RELATION_TYPES.items():
    print(f"  {name:<14} {source} -> {target}")

In [ ]:
# The boundary, tested. A registry-owned type is rejected with its own reason, so
# you can see the prompt holding rather than guess.
payload = {"entities": [
    {"name": "tumour biopsy", "type": "Procedure", "detail": "core needle"},
    {"name": "Week 12", "type": "Timepoint", "detail": ""},
    {"name": "Gilead Sciences", "type": "Sponsor", "detail": ""},
    {"name": "the appendix", "type": "Document", "detail": ""}],
 "relations": [
    {"source": "tumour biopsy", "target": "Week 12", "type": "PERFORMED_AT"},
    {"source": "Week 12", "target": "tumour biopsy", "type": "PERFORMED_AT"}]}

entities, relations, rejected = schema.validate(payload)
print(f"kept {len(entities)} entities, {len(relations)} relations")
for reason in rejected:
    print(f"  rejected: {reason}")

---
## 5. Extract and load

One call per chunk, cached by `(model, prompt, text)`. The prompt is in the key
because changing the schema changes the output.

In [ ]:
# Gated by RUN_EXTRACTION (config.py) — off by default, since this is the one
# layer with a real, ongoing dollar cost. This single call is cheap either way
# (one chunk), but it is here as a preview of extract_all() below, so it is
# gated the same way rather than running by itself while the real batch stays
# off — seeing one result and then not being able to run the rest is more
# confusing than seeing neither.
if config.RUN_EXTRACTION:
    sample = max(records, key=lambda c: int(c.get("n_tokens", 0)))
    print(sample["text"][:600], "\n" + "=" * 78)

    one = extract.extract_chunk(sample["text"], sample["chunk_id"])
    for entity in one["entities"]:
        print(f"  {entity['type']:<12} {entity['name']}")
    for relation in one["relations"]:
        print(f"  ({relation['source']}) -[{relation['type']}]-> ({relation['target']})")
    for reason in one["rejected"]:
        print(f"  rejected: {reason}")
else:
    print("RUN_EXTRACTION=0 — skipping. Set it to 1 in .env to run extraction "
          "(roughly $1.50 across a 3,000-chunk corpus at the gpt-4o-mini "
          "default — see EXTRACT_MODEL in config.py).")


In [ ]:
extractions = extract.extract_all(records) if config.RUN_EXTRACTION else []


In [ ]:
# Chunk nodes already exist from layer 1, so this only MATCHes them. An
# extraction referring to a chunk that was never loaded links nothing, rather
# than creating a Chunk with no document and no section.
if config.RUN_EXTRACTION:
    loaded = store.load(session, extractions)
else:
    print("RUN_EXTRACTION=0 — nothing to load. The graph has structure"
          + (" and registry" if config.RUN_REGISTRY else "")
          + " only.")


In [ ]:
stats = store.summary(session)

print("nodes by label:")
for label, count in stats["nodes"].items():
    print(f"  {label:<20} {count}")
print("\nrelationships:")
for kind, count in stats["relationships"].items():
    print(f"  {kind:<20} {count}")

# Provenance, which is the point of building it this way.
print("\nnodes by source:")
for row in session.run("""
    MATCH (n) WHERE n.source IS NOT NULL
    RETURN n.source AS source, count(*) AS n ORDER BY n DESC
"""):
    print(f"  {row['source']:<20} {row['n']}")

# Aura's free tier caps the instance at 200,000 nodes and 400,000 relationships.
# Past that, writes fail rather than slow down.
nodes = sum(stats["nodes"].values())
rels = sum(stats["relationships"].values())
print(f"\ntotal: {nodes:,} nodes, {rels:,} relationships")
if nodes > 150_000 or rels > 300_000:
    print("approaching the Aura free tier limit (200k nodes / 400k relationships)")

---
## 6. Score the extractor against ground truth

This is what having both layers makes possible, and almost no graph pipeline can do
it: the registry is ground truth, so extraction can be **measured** rather than
trusted.

The probe deliberately extracts fields the production schema excludes — sponsor,
phase, condition, enrolment. Measuring where the answer is known, then carrying the
result across to the fields where it is not.

In [ ]:
# One entry per document, using its opening text — where a protocol states its
# sponsor and phase. Giving the extractor its best chance is the honest way to
# measure it: a low score on favourable input means something.
by_document = {}
for chunk in sorted(records, key=lambda c: int(c.get("position", 0))):
    doc_id = chunk.get("doc_id")
    by_document.setdefault(doc_id, []).append(chunk.get("text", ""))

documents = []
for doc_id, texts in by_document.items():
    nct_id = structure.find_nct_id(doc_id, "", " ".join(texts[:3]))
    if nct_id:
        documents.append({"doc_id": doc_id, "nct_id": nct_id,
                          "text": " ".join(texts[:6])})

print(f"{len(documents)} documents with a registry trial to score against\n")
scored = accuracy.score(documents)

### Reading the score

**exact** — the extracted value matches the registry after normalising for
formatting.

**partial** — one contains the other. "Gilead" against "Gilead Sciences" is a
near-miss, not the same failure as answering "Pfizer", and collapsing the two would
hide the difference between a formatting gap and a factual error.

**wrong** — a different answer. This is the number that should worry you.

**missing** — the excerpt did not state it. Not an extractor failure.

Accuracy counts partial as half.

If sponsor and phase score below about 0.8 on their most favourable input, treat
every extracted claim in layer 3 with the same scepticism — it is the same model on
harder material, with no ground truth to check it against.

The graph is built. Continue in `02_query_graph.ipynb`.